### Biomedical Knowledge Graph & Contrastive Path Explorer
This notebook downloads **Hetionet v1.0** (a lightweight biomedical knowledge graph covering diseases, symptoms, anatomy, genes, and compounds) and visualizes the clinical path from clinical clues to diagnoses.

### 1. Imports and Downloads

In [ ]:
import os
import json
import urllib.request
import gzip
from pathlib import Path
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import subprocess
import sys
from tqdm import tqdm
import gdown
import pickle
import pandas as pd
import re
from openrouter import OpenRouter
print("Libraries imported successfully!")

Libraries imported successfully!


In [14]:
DATA_DIR = "../data/drknows_data"
os.makedirs(DATA_DIR, exist_ok=True)

NODES_URL = "https://github.com/hetio/hetionet/raw/main/hetnet/tsv/hetionet-v1.0-nodes.tsv"
EDGES_URL = "https://github.com/hetio/hetionet/raw/main/hetnet/tsv/hetionet-v1.0-edges.sif.gz"

# 1. Download the DRKnows UMLS/SNOMED Graph (~700 MB)
graph_path = os.path.join(DATA_DIR, "drknows_graph.pkl")
if not os.path.exists(graph_path):
    print("Downloading DRKnows graph object (~700 MB)...")
    gdown.download(id="1zlb0zey_tAnFWtCY_NvhA0dqfydL4Ph7", output=graph_path, quiet=False)
else:
    print(f"DRKnows graph already downloaded: {graph_path}")

# 2. Download the CUI-Preferred English Text Vocabulary
vocab_path = os.path.join(DATA_DIR, "cui_vocab.pkl")
if not os.path.exists(vocab_path):
    print("Downloading CUI text vocabulary (~15 MB)...")
    gdown.download(id="1xnZyz_ePAcXzmzCaqJHsAI0sf8LsG8DA", output=vocab_path, quiet=False)
else:
    print(f"CUI vocabulary already downloaded: {vocab_path}")
# 3. Download the 108 physician-selected relation types
relations_path = os.path.join(DATA_DIR, "relations.csv")
if not os.path.exists(relations_path):
    import urllib.request
    import json
    import base64
    
    print("Fetching UMLS refined relations list...")
    url = "https://api.github.com/repos/serenayj/DRKnows/contents/UMLS_refined_relations_DRKnows.csv"
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as resp:
        data = json.loads(resp.read().decode("utf-8"))
        content = base64.b64decode(data["content"]).decode("utf-8")
        with open(relations_path, "w") as f:
            f.write(content)
else:
    print(f"Relations list already downloaded: {relations_path}")
    
print("\nAll DRKnows data files downloaded and ready!")


DRKnows graph already downloaded: ../data/drknows_data/drknows_graph.pkl
CUI vocabulary already downloaded: ../data/drknows_data/cui_vocab.pkl
Relations list already downloaded: ../data/drknows_data/relations.csv

All DRKnows data files downloaded and ready!


In [15]:
# 1. Load the 108 Clinical Relations
print("1. Loading Relations...")
df_relations = pd.read_csv(relations_path)
print(f"   Loaded {len(df_relations)} curated diagnostic relations.")
print("   Sample relations:", df_relations['Relation'].head(5).tolist())

# 2. Load the CUI English Vocabulary
print("\n2. Loading CUI English Vocabulary...")
with open(vocab_path, "rb") as f:
    cui_vocab = pickle.load(f)

print(f"   Vocabulary type: {type(cui_vocab)}")
if isinstance(cui_vocab, dict):
    print(f"   Total medical terms in vocabulary: {len(cui_vocab):,}")
    sample_cui, sample_name = next(iter(cui_vocab.items()))
    print(f"   Sample entry: {sample_cui} ➔ '{sample_name}'")
elif isinstance(cui_vocab, list):
    print(f"   Total items: {len(cui_vocab):,}")
    print(f"   Sample items: {cui_vocab[:3]}")

# 3. Load the Knowledge Graph
print("\n3. Loading DRKnows Graph Object...")
with open(graph_path, "rb") as f:
    dr_graph = pickle.load(f)

print(f"   Graph object type: {type(dr_graph)}")

# Check graph properties (NetworkX vs Dict vs PyTorch)
if hasattr(dr_graph, "number_of_nodes"):
    print(f"   ✅ It's a NetworkX graph!")
    print(f"   Nodes: {dr_graph.number_of_nodes():,}")
    print(f"   Edges: {dr_graph.number_of_edges():,}")
elif isinstance(dr_graph, dict):
    print(f"   Dictionary structure with keys: {list(dr_graph.keys())[:5]}")
else:
    print(f"   Available attributes: {[a for a in dir(dr_graph) if not a.startswith('_')][:8]}")

print("\n" + "=" * 50)
print("All three components loaded! Ready for path search.")

1. Loading Relations...
   Loaded 107 curated diagnostic relations.
   Sample relations: ['has measured component', 'same as', 'isa', 'component of', 'method of']

2. Loading CUI English Vocabulary...
   Vocabulary type: <class 'dict'>
   Total medical terms in vocabulary: 489,354
   Sample entry: C0238502 ➔ '[['A18691089', 'prolapse of urethra'], ['A3648946', 'Prolapse of urethra (disorder)'], ['A9341324', 'urethral prolapse'], ['A18616889', 'urethrocele'], ['A4448523', 'prolapse; urinary meatus'], ['A14149161', 'urethral prolapse'], ['A13560475', 'urethrocele (diagnosis)'], ['A4466493', 'urethra; prolapse'], ['A2950407', 'Urethrocele'], ['A18672592', 'urethra prolapse'], ['A0468309', 'URETHRA, PROLAPSE'], ['A17853588', 'Urethrocele'], ['A1862401', 'Prolapse;urethral'], ['A13651226', 'urethrocele'], ['A13809400', 'urethrocele (physical finding)'], ['A0468350', 'URETHROCELE'], ['A8352998', 'Urethrocele'], ['A20162504', 'Urethrocele'], ['A24681653', 'Urethrocele'], ['A25707233', 'Urethr

In [16]:
# 1. Build fast bidirectional mappings: CUI <-> English
cui_to_name = {}
name_to_cui = {}

print("Building fast English search index for 400k concepts...")
for cui, entries in cui_vocab.items():
    if isinstance(entries, list) and len(entries) > 0:
        # Take the cleanest primary name (second element in the list)
        primary_name = entries[0][1].strip() if len(entries[0]) > 1 else str(entries[0])
        cui_to_name[cui] = primary_name
        name_to_cui[primary_name.lower()] = cui
    elif isinstance(entries, str):
        cui_to_name[cui] = entries
        name_to_cui[entries.lower()] = cui

print(f"Indexed {len(cui_to_name):,} unique medical concept names!")

# 2. Plain English Search Function
def search_concept(query: str, top_k: int = 5):
    """Searches DRKnows vocabulary for matching medical terms and returns (CUI, Name)."""
    q = query.lower().strip()
    results = []
    
    # Check exact match first
    if q in name_to_cui:
        cui = name_to_cui[q]
        results.append((cui, cui_to_name[cui]))
    
    # Substring search
    for name, cui in name_to_cui.items():
        if q in name and (cui, cui_to_name[cui]) not in results:
            results.append((cui, cui_to_name[cui]))
            if len(results) >= top_k:
                break
                
    return results

# Quick test search:
print("\nSearch Test:")
print("Search 'chest pain':", search_concept("chest pain"))
print("Search 'myocardial infarction':", search_concept("myocardial infarction"))
print("Search 'hypertension':", search_concept("hypertension"))

Building fast English search index for 400k concepts...
Indexed 489,354 unique medical concept names!

Search Test:
Search 'chest pain': [('C0008031', '[D]Chest pain (situation)'), ('C0429692', 'Chest pain rating (staging scale)'), ('C0262384', 'Atypical chest pain'), ('C1279523', '[D]Central chest pain'), ('C0423636', 'Cardiac chest pain')]
Search 'myocardial infarction': [('C2926063', 'Myocardial infarction'), ('C0340305', 'Myocardial Infarctions, Diaphragmatic'), ('C5396562', 'Acute STEMI (ST elevation myocardial infarction) due to AM (acute marginal) branch of RCA (right coronary artery) occlusion'), ('C0155630', 'Acute myocardial infarction of anterolateral wall, subsequent episode of care'), ('C0155643', 'Acute myocardial infarction, of other inferior wall')]
Search 'hypertension': [('C1963138', 'Hypertension'), ('C0742524', 'COCAINE INDUCED HYPERTENSION'), ('C0152170', 'transient hypertension'), ('C1391599', 'c.pregnancy; nephritis, with secondary hypertension'), ('C4509210', '(

In [17]:
print(f"Total relations in DRKnows: {len(df_relations)}")

# Filter out non-clinical administrative/temporal shortcut edges
BANNED_RELATIONS = {
    'temporal context of',
    'has temporal context',
    'has subject relationship context',
    'subject relationship context of',
    'associated finding of', # often links to 'No cardiac failure' negations
}

BANNED_NODE_SUBSTRINGS = {
    'body system',
    'clinical investigation',
    'laboratory procedure',
    'evaluation procedure',
    'procedure by site',
    'entire body',
    'general finding',
    'context-dependent',
    'qualifier value',
}

GENERIC_ROOT_NAMES = {
    'procedure (procedure)',
    'functional observable',
    'clinical investigation',
    'laboratory procedure',
    'evaluation procedure',
    'observable entity',
    'finding by site',
    'regime/therapy',
    'action (qualifier value)',
}

# Keep only valid clinical reasoning relations
valid_relations = set(df_relations['Relation']) - BANNED_RELATIONS
print(f"Active clinical relations after filtering: {len(valid_relations)}")

Total relations in DRKnows: 107
Active clinical relations after filtering: 103


In [18]:
# 1. Filter our lookup index so it ONLY indexes nodes that exist inside dr_graph
graph_nodes_set = set(dr_graph.nodes())

# Build mapping of lowercase name -> CUI, strictly for active graph nodes
active_name_to_cui = {}
for cui, entries in cui_vocab.items():
    if cui in graph_nodes_set:
        if isinstance(entries, list) and len(entries) > 0:
            for entry in entries:
                if len(entry) > 1:
                    clean_name = entry[1].lower().strip()
                    if clean_name not in active_name_to_cui:
                        active_name_to_cui[clean_name] = cui
        elif isinstance(entries, str):
            clean_name = entries.lower().strip()
            if clean_name not in active_name_to_cui:
                active_name_to_cui[clean_name] = cui

print(f"Indexed {len(active_name_to_cui):,} valid terms mapped to active graph nodes!")

# 2. Robust Search Function
def search_active_concept(query: str, top_k: int = 5):
    """Finds concept matches that guaranteed exist in dr_graph."""
    q = query.lower().strip()
    results = []
    
    # 1. Exact match
    if q in active_name_to_cui:
        cui = active_name_to_cui[q]
        results.append((cui, cui_to_name.get(cui, q)))
        
    # 2. Substring match
    for name, cui in active_name_to_cui.items():
        if q in name and (cui, cui_to_name.get(cui, name)) not in results:
            results.append((cui, cui_to_name.get(cui, name)))
            if len(results) >= top_k:
                break
                
    return results

# 3. Updated Path Finder
def find_drknows_path(source_term: str, target_term: str):
    src_matches = search_active_concept(source_term, top_k=1)
    tgt_matches = search_active_concept(target_term, top_k=1)

    if not src_matches or not tgt_matches:
        print(f"❌ Could not resolve active nodes for '{source_term}' or '{target_term}'")
        return None

    src_cui, src_name = src_matches[0]
    tgt_cui, tgt_name = tgt_matches[0]

    try:
        # Check forward path in DiGraph
        raw_path = nx.shortest_path(dr_graph, source=src_cui, target=tgt_cui)
    except nx.NetworkXNoPath:
        try:
            # Check bidirectional clinical reachability
            raw_path = nx.shortest_path(dr_graph.to_undirected(), source=src_cui, target=tgt_cui)
        except nx.NetworkXNoPath:
            print(f"No path found between '{src_name}' and '{tgt_name}'")
            return None

    # Format steps using the 'label' attribute
    path_steps = []
    for i in range(len(raw_path)):
        node_cui = raw_path[i]
        node_name = cui_to_name.get(node_cui, node_cui)
        
        relation = ""
        if i < len(raw_path) - 1:
            next_cui = raw_path[i+1]
            edge_data = dr_graph.get_edge_data(node_cui, next_cui) or dr_graph.get_edge_data(next_cui, node_cui) or {}
            # DRKnows stores the relation in the 'label' key!
            relation = edge_data.get('label', 'associated_with')

        path_steps.append({
            "cui": node_cui,
            "name": node_name,
            "relation_to_next": str(relation)
        })

    return path_steps

# 4. Helper to print the path
def print_drknows_path(path_steps):
    if not path_steps:
        return
    print("\n" + "=" * 65)
    print("CLINICAL PATHWAY (DRKnows):")
    for i, step in enumerate(path_steps):
        if i == len(path_steps) - 1:
            print(f"  🏁 {step['name']} ({step['cui']})")
        else:
            print(f"  🔹 {step['name']} ({step['cui']})")
            print(f"      │  [{step['relation_to_next']}]")
            print(f"      ▼")
    print("=" * 65)


# Create a filtered graph view that ignores edges with banned shortcut relations
def is_valid_clinical_edge(u, v):
    edge_data = dr_graph.get_edge_data(u, v) or {}
    rel = edge_data.get('label', '')
    return rel not in BANNED_RELATIONS

def is_valid_clinical_node(node):
    """Filters out abstract taxonomy roots without blocking specific medical terms."""
    name = cui_to_name.get(node, '').lower().strip()
    
    # Check exact generic root match
    if name in GENERIC_ROOT_NAMES:
        return False
        
    # Check substring matches
    for banned in BANNED_NODE_SUBSTRINGS:
        if banned in name:
            return False
            
    return True

# Subgraph view restricting traversal strictly to real medical edges
clinical_G = nx.subgraph_view(
    dr_graph, 
    filter_node=is_valid_clinical_node, 
    filter_edge=is_valid_clinical_edge
)

print(f"Clean Clinical Graph View Active!")

def find_clinical_path(source_term: str, target_term: str):
    """Finds pure clinical path without temporal/administrative shortcuts."""
    src_matches = search_active_concept(source_term, top_k=1)
    tgt_matches = search_active_concept(target_term, top_k=1)

    if not src_matches or not tgt_matches:
        print(f"❌ Could not resolve: '{source_term}' or '{target_term}'")
        return None

    src_cui, src_name = src_matches[0]
    tgt_cui, tgt_name = tgt_matches[0]

    try:
        # Search path strictly on filtered clinical view
        raw_path = nx.shortest_path(clinical_G, source=src_cui, target=tgt_cui)
    except nx.NetworkXNoPath:
        try:
            raw_path = nx.shortest_path(clinical_G.to_undirected(), source=src_cui, target=tgt_cui)
        except nx.NetworkXNoPath:
            print(f"No clinical path between '{src_name}' and '{tgt_name}'")
            return None

    path_steps = []
    for i in range(len(raw_path)):
        node_cui = raw_path[i]
        node_name = cui_to_name.get(node_cui, node_cui)
        relation = ""
        if i < len(raw_path) - 1:
            next_cui = raw_path[i + 1]
            edge_data = clinical_G.get_edge_data(node_cui, next_cui) or clinical_G.get_edge_data(next_cui, node_cui) or {}
            relation = edge_data.get('label', 'associated_with')

        path_steps.append({
            "cui": node_cui,
            "name": node_name,
            "relation": relation
        })

    return path_steps

Indexed 1,462,169 valid terms mapped to active graph nodes!
Clean Clinical Graph View Active!


In [19]:
for clue, target in [("shortness of breath", "heart failure"), ("polyuria", "diabetes mellitus"), ("hemoptysis", "tuberculosis"), ("proteinuria", "nephrotic syndrome"), ("neck stiffness", "meningitis"), ("joint pain", "gout")]:
    print(f"\nQuery: '{clue}' ➔ '{target}'")
    path = find_clinical_path(clue, target)
    if path:
        for i in range(len(path) - 1):
            print(f"  • {path[i]['name']} ──[{path[i]['relation']}]──► {path[i+1]['name']}")


Query: 'shortness of breath' ➔ 'heart failure'
  • Breathlessness NOS (finding) ──[finding site of]──► Respiratory tract
  • Respiratory tract ──[has finding site]──► Malignant neoplasm of heart
  • Malignant neoplasm of heart ──[finding site of]──► Heart structure
  • Heart structure ──[has finding site]──► Heart failure

Query: 'polyuria' ➔ 'diabetes mellitus'
  • Polyuric state, NOS ──[isa]──► Dipsogenic Diabetes Insipidus
  • Dipsogenic Diabetes Insipidus ──[finding site of]──► Entire endocrine gonad (body structure)
  • Entire endocrine gonad (body structure) ──[has finding site]──► Diabetes Mellitus

Query: 'hemoptysis' ➔ 'tuberculosis'
  • Haemoptysis ──[finding site of]──► Respiratory system subdivision (body structure)
  • Respiratory system subdivision (body structure) ──[has finding site]──► Late effects of respiratory tuberculosis
  • Late effects of respiratory tuberculosis ──[temporally followed by]──► Tuberculosis

Query: 'proteinuria' ➔ 'nephrotic syndrome'
  • Protein

### 2. Multi-clue subgraph

In [20]:
def find_convergence_web(clues_list: list, target_term: str):
    tgt_matches = search_active_concept(target_term, top_k=1)
    if not tgt_matches or tgt_matches[0][0] not in clinical_G:
        print(f"❌ Target '{target_term}' not in active clinical graph.")
        return None
    tgt_cui, tgt_name = tgt_matches[0]

    resolved_clues = []
    for c in clues_list:
        matches = search_active_concept(c, top_k=1)
        if matches and matches[0][0] in clinical_G:
            resolved_clues.append((matches[0][0], matches[0][1], c))
        else:
            print(f"⚠️ Clue '{c}' not resolved in active graph.")

    if not resolved_clues:
        print("No clues resolved.")
        return None

    web_edges = []
    clue_nodes = set()
    intermediate_nodes = set()
    path_details = {}

    for c_cui, c_name, raw_query in resolved_clues:
        clue_nodes.add(c_cui)
        try:
            raw_path = nx.shortest_path(clinical_G, source=c_cui, target=tgt_cui)
        except (nx.NetworkXNoPath, nx.NodeNotFound):
            try:
                raw_path = nx.shortest_path(clinical_G.to_undirected(), source=c_cui, target=tgt_cui)
            except (nx.NetworkXNoPath, nx.NodeNotFound):
                raw_path = []

        if raw_path:
            path_details[c_name] = []
            for i in range(len(raw_path) - 1):
                u = raw_path[i]
                v = raw_path[i + 1]
                edge_data = clinical_G.get_edge_data(u, v) or clinical_G.get_edge_data(v, u) or {}
                rel = edge_data.get('label', 'relates_to')
                web_edges.append((u, v, rel))
                path_details[c_name].append((cui_to_name.get(u, u), rel, cui_to_name.get(v, v)))
                if u != c_cui and u != tgt_cui:
                    intermediate_nodes.add(u)
                if v != c_cui and v != tgt_cui:
                    intermediate_nodes.add(v)
        else:
            path_details[c_name] = ["❌ No path to target"]

    all_web_nodes = clue_nodes.union(intermediate_nodes).union({tgt_cui})
    subG = clinical_G.subgraph(all_web_nodes).copy()

    return {
        "target": {"cui": tgt_cui, "name": tgt_name},
        "clues": resolved_clues,
        "clue_cuis": clue_nodes,
        "intermediate_cuis": intermediate_nodes,
        "subgraph": subG,
        "web_edges": web_edges,
        "path_details": path_details
    }


def plot_convergence_web(web_data, title="Clinical Convergence Web"):
    """Visualizes the multi-clue convergence web."""
    if not web_data or web_data['subgraph'].number_of_nodes() == 0:
        print("Nothing to plot.")
        return

    subG = web_data['subgraph']
    tgt_cui = web_data['target']['cui']
    clue_cuis = web_data['clue_cuis']

    # Color code: Clues = Sky Blue, Intermediates = Purple/Violet, Target = Emerald Green
    node_colors = []
    labels = {}
    for node in subG.nodes():
        name = cui_to_name.get(node, node)
        # Shorten label for clean plotting
        short_name = name.split('(')[0].strip()
        labels[node] = short_name

        if node == tgt_cui:
            node_colors.append('#10b981') # Emerald Target
        elif node in clue_cuis:
            node_colors.append('#38bdf8') # Sky Blue Clues
        else:
            node_colors.append('#a855f7') # Violet Intermediates

    # plt.figure(figsize=(11, 6))
    # pos = nx.spring_layout(subG, seed=42, k=0.9)

    # nx.draw_networkx_nodes(subG, pos, node_color=node_colors, node_size=2800, alpha=0.9)
    # nx.draw_networkx_edges(subG, pos, width=2, edge_color='#64748b', arrows=True, arrowsize=14)
    # nx.draw_networkx_labels(subG, pos, labels, font_size=8, font_weight="bold")

    # edge_labels = {(u, v): d.get('label', '') for u, v, d in subG.edges(data=True)}
    # nx.draw_networkx_edge_labels(subG, pos, edge_labels=edge_labels, font_size=7)

    # plt.title(f"{title} ➔ {web_data['target']['name']}", fontsize=12, fontweight='bold')
    # plt.axis("off")
    # plt.tight_layout()
    # plt.show()

    # Print Text Convergence Summary
    print("\n" + "=" * 65)
    print(f"🎯 TARGET DIAGNOSIS: {web_data['target']['name']}")
    print("=" * 65)
    for clue_name, steps in web_data['path_details'].items():
        if isinstance(steps[0], tuple):
            chain_str = " ➔ ".join([f"{s[0]}" for s in steps] + [steps[-1][2]])
            print(f"  🔹 Clue: '{clue_name}'")
            print(f"     Path: {chain_str}")
        else:
            print(f"  ❌ Clue: '{clue_name}' ➔ No path found")
    print("=" * 65)

In [21]:
multi_clue_cases = [
    # 1. Cardiology (Triad: Chest pain + Shortness of breath + Coronary Artery)
    {
        "clues": ["chest pain", "shortness of breath", "coronary artery"],
        "target": "myocardial infarction",
        "category": "Cardiology - Acute Coronary Presentation"
    },
    # 2. Endocrinology (Polyuria + Polydipsia / Thirst + Hyperglycemia)
    {
        "clues": ["polyuria", "thirst", "hyperglycemia"],
        "target": "diabetes mellitus",
        "category": "Endocrinology - Classic Diabetic Triad"
    },
    # 3. Pulmonology / Infectious Disease (Cough + Hemoptysis + Fever)
    {
        "clues": ["cough", "hemoptysis", "fever"],
        "target": "tuberculosis",
        "category": "Pulmonology - Cavitary Lung Infection"
    },
    # 4. Nephrology (Proteinuria + Edema + Hypoalbuminemia)
    {
        "clues": ["proteinuria", "edema", "hypoalbuminemia"],
        "target": "nephrotic syndrome",
        "category": "Nephrology - Glomerular Syndrome"
    },
    # 5. Neurology (Neck stiffness + Headache + Fever)
    {
        "clues": ["neck stiffness", "headache", "fever"],
        "target": "meningitis",
        "category": "Neurology - Classic Meningeal Triad"
    },
    # 6. Rheumatology (Joint pain + Redness + Metatarsophalangeal / Toe)
    {
        "clues": ["joint pain", "erythema", "metatarsophalangeal joint"],
        "target": "gout",
        "category": "Rheumatology - Podagra / Monoarthritis"
    }
]

# Run the 6 tests:
for case in multi_clue_cases:
    print(f"\n🔬 Running Case: [{case['category']}]")
    web = find_convergence_web(case["clues"], case["target"])
    if web:
        plot_convergence_web(web, title=case["category"])


🔬 Running Case: [Cardiology - Acute Coronary Presentation]

🎯 TARGET DIAGNOSIS: Myocardial infarction
  🔹 Clue: '[D]Chest pain (situation)'
     Path: [D]Chest pain (situation) ➔ Angina ➔ angina pectoris following myocardial infarction (diagnosis) ➔ Myocardial infarction
  🔹 Clue: 'Breathlessness NOS (finding)'
     Path: Breathlessness NOS (finding) ➔ Respiratory tract ➔ Chronic anoxic encephalopathy (disorder) ➔ Infarction, NOS ➔ Myocardial infarction
  🔹 Clue: 'Entire coronary artery'
     Path: Entire coronary artery ➔ Coronary artery structure (body structure) ➔ angina pectoris following myocardial infarction (diagnosis) ➔ Myocardial infarction

🔬 Running Case: [Endocrinology - Classic Diabetic Triad]

🎯 TARGET DIAGNOSIS: Diabetes Mellitus
  🔹 Clue: 'Polyuric state, NOS'
     Path: Polyuric state, NOS ➔ Dipsogenic Diabetes Insipidus ➔ Entire endocrine gonad (body structure) ➔ Diabetes Mellitus
  🔹 Clue: 'Desire for fluids'
     Path: Desire for fluids ➔ Increased thirst (finding)

### 3. Integrate LLM

In [ ]:

# Enter custom path for OpenRouter API key
key_path = Path("~/.keys/temp-usmle-graphgen-test.env").expanduser()

if not key_path.exists():
    raise FileNotFoundError(f"Key file not found at: {key_path}")

OPENROUTER_API_KEY = key_path.read_text().strip()

if not OPENROUTER_API_KEY:
    raise ValueError("OPENROUTER_API_KEY environment variable is not set")

def extract_clues_openrouter(
    vignette: str,
    correct_answer: str,
    api_key: str = OPENROUTER_API_KEY,
) -> list[str]:
    """
    Extracts the 3 most salient clinical findings (symptoms, signs, abnormal labs/vitals)
    from a vignette using OpenRouter's free tier models.
    """
    prompt = f"""You are a USMLE Pathophysiology Expert.
Given this medical vignette and the correct diagnosis/answer, extract the 3 most salient CLINICAL FINDINGS (hallmark symptoms, physical exam signs, anatomical locations, or abnormal lab/ECG findings) that lead directly to the diagnosis.

Strict Rules:
1. ONLY return specific medical entities (e.g. "chest pain", "diaphoresis", "ST segment elevation", "podagra", "nuchal rigidity").
2. DO NOT return generic modifiers or temporal adjectives (e.g. NEVER return "sudden onset", "acute", "chronic", "severe", "elderly").
3. Normalize physical exam descriptions into canonical clinical signs (e.g. neck flexion causing hip flexion -> "Brudzinski sign").
4. Return ONLY a JSON list of 3 strings. No markdown fences, no explanations.

VIGNETTE:
{vignette}

CORRECT ANSWER:
{correct_answer}

JSON:"""

    try:
        with OpenRouter(
            api_key=api_key,
            http_referer="http://localhost:8000",
            x_open_router_title="My app",
        ) as client:

            response = client.chat.send(
            model="meta-llama/llama-3.3-70b-instruct:free",
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
            # max_tokens=400,
            models=[
                "qwen/qwen-2.5-72b-instruct:free",
                "mistralai/mistral-small-24b-instruct-2501:free",
                "openrouter/auto:free",
            ],  # Removed 'openrouter/auto:free'
        )

        message = response.choices[0].message
        reasoning = getattr(message, "reasoning", "")
        content = message.content or ""
        resolved_model = getattr(response, "model", "unknown-model")

        # print(f"==== Message: {message or "empty"}")
        # print(f"==== Reasoning: {reasoning or "empty"}")
        # print(f"==== Content: {content or "empty"}")
        print(f"==== Model used: {resolved_model}")


        clean_text = content.strip()
        if "```" in clean_text:
            parts = clean_text.split("```")
            if len(parts) >= 2:
                clean_text = parts[1]
                if clean_text.startswith("json"):
                    clean_text = clean_text[4:]

        start_idx = clean_text.find("[")
        end_idx = clean_text.rfind("]")
        if start_idx != -1 and end_idx != -1 and end_idx > start_idx:
            json_substr = clean_text[start_idx : end_idx + 1]
            clues = json.loads(json_substr)
            if isinstance(clues, list) and len(clues) > 0:
                print(f"Success!")
                return clues

        print("Failed to parse a JSON array from response.")
        return content

    except Exception as e:
        print(f"OpenRouter API call or parsing failed: {e}")
        return []

In [45]:
vignette_1 = """
A 62-year-old man presents to the emergency department with severe, crushing substernal chest 
pain that began 1 hour ago while walking his dog. The pain radiates to his left shoulder and jaw. 
His blood pressure is 154/92 mm Hg, pulse is 112/min, and respirations are 22/min. He appears 
pale, anxious, and diaphoretic. ECG reveals ST-segment elevations in leads II, III, and aVF.
"""

vignette_2 = """
A 24-year-old college student is brought to the clinic by his roommate because of a severe headache, 
lethargy, and sensitivity to light. He has a temperature of 39.1°C (102.4°F), blood pressure of 
110/70 mm Hg, and pulse of 105/min. On physical examination, passive flexion of the neck results 
in spontaneous flexion of the hips and knees.
"""

vignette_3 = """
A 52-year-old male presents with sudden onset of excruciating pain and swelling in his right great toe 
that woke him up from sleep. He reports consuming a large steak dinner with several glasses of red wine 
the night before. Physical examination reveals an exquisitely tender, erythematous, and swollen 
first metatarsophalangeal joint.
"""

# Run the 3 test vignettes:
output1 = extract_clues_openrouter(vignette_1, "myocardial infarction")
print(f"Output 1: {output1}")

output2 = extract_clues_openrouter(vignette_2, "meningitis")
print(f"Output 2: {output2}")

output3 = extract_clues_openrouter(vignette_3, "gout")
print(f"Output 3: {output3}")

==== Model used: inclusionai/ling-3.0-flash-sante:free
Success from inclusionai/ling-3.0-flash-sante:free!
Output 1: ['crushing chest pain', 'ST elevations', 'diaphoresis']
==== Model used: inclusionai/ling-3.0-flash-sante:free
Success from inclusionai/ling-3.0-flash-sante:free!
Output 2: ['fever', 'headache', 'Brudzinski sign']
==== Model used: inclusionai/ling-3.0-flash-sante:free
Success from inclusionai/ling-3.0-flash-sante:free!
Output 3: ['great toe pain', 'first MTP swelling', 'erythema']


In [47]:
def run_end_to_end_test(vignette: str, correct_answer: str):
    """
    1. Uses OpenRouter to extract the 3 clinical clues from the vignette.
    2. Feeds clues + correct answer into DRKnows graph convergence engine.
    3. Prints the verified pathophysiological funnel.
    """
    print("=" * 65)
    print("📋 CLINICAL VIGNETTE:")
    print(vignette.strip())
    print(f"\n🎯 TARGET DIAGNOSIS: {correct_answer}")
    print("=" * 65)

    # Step 1: OpenRouter Extraction
    clues = extract_clues_openrouter(vignette, correct_answer)
    print(f"\n✨ Extracted Clues from LLM: {clues}")

    if not clues:
        print("Could not extract clues. Skipping graph convergence.")
        return

    # Step 2: DRKnows Convergence Web
    print("\n🕸️ Searching DRKnows Pathophysiology Paths...")
    web = find_convergence_web(clues, correct_answer)

    if web:
        print("\n" + "=" * 65)
        print(f"🎯 VERIFIED CONVERGENCE FUNNEL ➔ {web['target']['name']}")
        print("=" * 65)
        for clue_name, steps in web['path_details'].items():
            if isinstance(steps[0], tuple):
                chain_str = " ➔ ".join([f"{s[0]}" for s in steps] + [steps[-1][2]])
                print(f"  🔹 Clue: '{clue_name}'")
                print(f"     Path: {chain_str}")
            else:
                print(f"  ❌ Clue: '{clue_name}' ➔ No path found to target")
        print("=" * 65)
    else:
        print("❌ DRKnows failed to find paths for these clues.")

In [48]:
vignette_1 = """
A 62-year-old man presents to the emergency department with severe, crushing substernal chest 
pain that began 1 hour ago while walking his dog. The pain radiates to his left shoulder and jaw. 
His blood pressure is 154/92 mm Hg, pulse is 112/min, and respirations are 22/min. He appears 
pale, anxious, and diaphoretic. ECG reveals ST-segment elevations in leads II, III, and aVF.
"""

vignette_2 = """
A 24-year-old college student is brought to the clinic by his roommate because of a severe headache, 
lethargy, and sensitivity to light. He has a temperature of 39.1°C (102.4°F), blood pressure of 
110/70 mm Hg, and pulse of 105/min. On physical examination, passive flexion of the neck results 
in spontaneous flexion of the hips and knees.
"""

vignette_3 = """
A 52-year-old male presents with sudden onset of excruciating pain and swelling in his right great toe 
that woke him up from sleep. He reports consuming a large steak dinner with several glasses of red wine 
the night before. Physical examination reveals an exquisitely tender, erythematous, and swollen 
first metatarsophalangeal joint.
"""

# Run the 3 test vignettes:
print("\n🔬 TEST 1: CARDIOLOGY VIGNETTE")
run_end_to_end_test(vignette_1, "myocardial infarction")

print("\n\n🔬 TEST 2: NEUROLOGY / INFECTIOUS DISEASE VIGNETTE")
run_end_to_end_test(vignette_2, "meningitis")

print("\n\n🔬 TEST 3: RHEUMATOLOGY VIGNETTE")
run_end_to_end_test(vignette_3, "gout")


🔬 TEST 1: CARDIOLOGY VIGNETTE
📋 CLINICAL VIGNETTE:
A 62-year-old man presents to the emergency department with severe, crushing substernal chest 
pain that began 1 hour ago while walking his dog. The pain radiates to his left shoulder and jaw. 
His blood pressure is 154/92 mm Hg, pulse is 112/min, and respirations are 22/min. He appears 
pale, anxious, and diaphoretic. ECG reveals ST-segment elevations in leads II, III, and aVF.

🎯 TARGET DIAGNOSIS: myocardial infarction
==== Model used: inclusionai/ling-3.0-flash-sante:free
Success!

✨ Extracted Clues from LLM: ['crushing chest pain', 'diaphoresis', 'ST elevations']

🕸️ Searching DRKnows Pathophysiology Paths...
⚠️ Clue 'ST elevations' not resolved in active graph.

🎯 VERIFIED CONVERGENCE FUNNEL ➔ Myocardial infarction
  🔹 Clue: 'Crushing chest pain'
     Path: Crushing chest pain ➔ Structure of nervous system ➔ Pain ➔ angina pectoris following myocardial infarction (diagnosis) ➔ Myocardial infarction
  🔹 Clue: '[D]Diaphoresis (situa